# protein-generation — stage1 ベースラインを Colab で回す

上から順に実行するだけで、`build_dataset.py` → `train_stage1.py` が走ります。

- `build_dataset.py`: HF からデータ取得 → MMseqs2 で 50% クラスタリング → 代表配列を `data/` に、埋め込みを `outputs/embeddings/` に書き出し
- `train_stage1.py`: 代表配列だけを使って ProLLaMA_Stage_1 を LoRA 学習（ProTrek ベクトルを使わない比較用ベースライン）

前提:

- GPU ランタイム（T4 16GB 以上。4bit 量子化なので 8GB 級でも可）
- ディスク 30GB 以上（モデル約 14GB + データ約 2.5GB + 出力）
- データ準備は MMseqs2 クラスタリングが支配的で 1〜3 時間。学習は step 数次第

## 0. コードの取得

GitHub にスクリプトが push 済みなら clone します。`/content/protein-generation/` に
ファイルをアップロード済みならそれを使います（`origin/main` にスクリプトが無い場合は、
先に push するか、左のファイルペインから配置してください）。

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/yosshstd/protein-generation.git"
PROJECT = Path("/content/protein-generation")
REQUIRED = [
    "build_dataset.py", "train_stage1.py", "train.py", "config.yaml",
    "pyproject.toml", "uv.lock", "tools/get_mmseqs.sh",
]

if not all((PROJECT / name).exists() for name in REQUIRED):
    if (PROJECT / ".git").exists():
        subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)

missing = [name for name in REQUIRED if not (PROJECT / name).exists()]
assert not missing, f"不足しています: {missing}（GitHub に push するか、ファイルを配置してください）"
os.chdir(PROJECT)
print("cwd:", os.getcwd())

In [ ]:
# キャッシュを /content 側に置いて、コンテナの / を圧迫しないようにします。
# 以降の ! コマンドはこのカレントディレクトリで実行されます。
import os
os.environ["HF_HOME"] = "/content/.hf-cache"
os.environ["UV_CACHE_DIR"] = "/content/.uv-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("HF_HOME =", os.environ["HF_HOME"])

## 1. 依存パッケージのインストール

`pyproject.toml` + `uv.lock` から、Python 3.13 の venv を作ります（`uv sync`）。
torch は cu129 固定なので、Colab のドライバが古いと `cuda: False` になります（下の注意参照）。

In [ ]:
%%bash
set -euo pipefail
cd /content/protein-generation
pip install -q uv
uv --version
uv python install 3.13
uv sync --frozen --python 3.13
uv run python -c "import torch; print(torch.__version__, 'cuda:', torch.cuda.is_available())"

In [ ]:
# 公開リポジトリなら不要。gated のときだけ、Colab の Secrets に HF_TOKEN を登録して実行。
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
except Exception as exc:
    print("HF_TOKEN なしで続行:", exc)

## 2. MMseqs2 の取得

約 32MB の静的ビルドを `tools/mmseqs/` に入れます（`build_dataset.py` が使います）。

In [ ]:
!cd /content/protein-generation && bash tools/get_mmseqs.sh

## 3. データセットを作る

`ids.tsv`（203MB）+ `sequence.index`（2.2GB）を取得し、569,792 配列を 50% 同一性で
クラスタリングして代表 118,576 配列を書き出します。クラスタリングが支配的です。

中断しても再実行すれば、済んだステップ（ダウンロード / クラスタリング）はスキップされます。

In [ ]:
# --threads は実行環境の CPU コア数。無料枠は 2 コア程度なので時間がかかります。
!cd /content/protein-generation && uv run python build_dataset.py --threads $(nproc)

In [ ]:
!cd /content/protein-generation && ls -lh data/protrek650m_swissprot_50pct_reps.parquet outputs/embeddings/ && du -sh data work outputs

## 4. stage1 ベースラインの学習

まず 3 step だけ回して経路を確認し、そのあと本番を回します。
本番の 1 epoch は約 55,500 step（batch 1 × grad accum 2、111,002 サンプル）なので、
Colab の 12 時間制限に収めるには step 数を切るのが現実的です。

In [ ]:
# 動作確認（出力は smoke 用の別ディレクトリへ）
!cd /content/protein-generation && uv run python train_stage1.py --max-steps 3 dataset.limit=200 output_dir=outputs/stage1_smoke

In [ ]:
# 本番。MAX_STEPS は学習したい step 数に合わせて変更してください。
MAX_STEPS = 2000
!cd /content/protein-generation && uv run python train_stage1.py --max-steps {MAX_STEPS} output_dir=outputs/stage1_colab

# 全件（時間が足りる場合のみ）:
# !cd /content/protein-generation && uv run python train_stage1.py output_dir=outputs/stage1_colab

In [ ]:
# セッションが切れたあとの再開。output_dir と設定を揃えて --resume を付けます。
# --max-steps は「合計」step 数なので、続きを 4000 まで進める場合は 4000 を指定。
!cd /content/protein-generation && uv run python train_stage1.py --resume --max-steps 4000 output_dir=outputs/stage1_colab

In [ ]:
# セッション終了で /content は消えるため、学習結果だけ Drive に退避します。
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/protein-generation
!cp -r outputs/stage1_colab data/protrek650m_swissprot_50pct_reps.parquet /content/drive/MyDrive/protein-generation/
!ls -lh /content/drive/MyDrive/protein-generation

## トラブルシューティング

- **`torch.cuda.is_available()` が False**: torch は cu129 固定です。Colab のドライバが
  cu129 より古い場合に起きます。`pyproject.toml` の `[[tool.uv.index]]` を実行環境の
  CUDA に合わせて変更してから `uv lock` → `uv sync` し直してください。
- **CUDA OOM**: `model.max_seq_length=640` のように下げて実行します
  （例: `!cd /content/protein-generation && uv run python train_stage1.py --max-steps 3 model.max_seq_length=640`）。
- **`missing: ... (run build_dataset.py first)`**: 先に §3 を完走させてください。
- **`MMseqs2 not found`**: §2 を実行するか `--mmseqs` でパスを指定してください。
- **セッション切断**: §4 の resume セルを実行してください（`save_steps: 500` ごとにチェックポイント）。